In [0]:
# 04_eval_seed — eval/ground_truth_case.jsonl → access_drift.eval.ground_truth_case 적재
# 주의: 탐지 파이프라인(ingest/silver/intermediate/gold)은 이 테이블을 입력으로 사용하지 않는다.

In [0]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("eval_seed_path", "data/dev/eval/ground_truth_case.jsonl")

CATALOG = dbutils.widgets.get("catalog")
EVAL_SEED_RELATIVE_PATH = dbutils.widgets.get("eval_seed_path").strip("/")
EVAL_SEED_LOCAL_PATH = f"/Workspace/Shared/access-drift/files/{EVAL_SEED_RELATIVE_PATH}"
EVAL_SEED_PATH = f"file:{EVAL_SEED_LOCAL_PATH}"


In [0]:
import os

if not os.path.isfile(EVAL_SEED_LOCAL_PATH):
    raise RuntimeError(
        f"Eval seed 파일을 찾을 수 없습니다: {EVAL_SEED_LOCAL_PATH}\n"
        "bundle deploy가 완료됐는지 확인하세요."
    )
print(f"eval seed path OK: {EVAL_SEED_LOCAL_PATH}")

In [0]:
from pyspark.sql import functions as F

df = (spark.read.json(EVAL_SEED_PATH)
      .withColumn("created_at", F.to_timestamp("created_at")))

(df.select(
    "case_id","scenario_id","scenario_family",
    "subject_type","subject_id","asset_id",
    "drift_type","is_planted_risk","is_distractor",
    "pattern","expected_severity","seed_finding_type","label_reason","created_at"
).write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.eval.ground_truth_case"))

count = spark.table(f"{CATALOG}.eval.ground_truth_case").count()
assert count > 0, "ground_truth_case has no rows"
print(f"ground_truth_case: {count} rows")
spark.table(f"{CATALOG}.eval.ground_truth_case").show(truncate=False)

positive_count = spark.table(f"{CATALOG}.eval.ground_truth_case").where("is_planted_risk = true").count()
distractor_count = spark.table(f"{CATALOG}.eval.ground_truth_case").where("is_distractor = true").count()
normal_count = spark.table(f"{CATALOG}.eval.ground_truth_case").where("is_planted_risk = false AND is_distractor = false").count()
print(f"positive={positive_count}, distractor={distractor_count}, normal={normal_count}")


## eval ↔ Silver 매칭 검증
# subject_id = silver_principals.source_principal_id 로 조인 가능해야 함
# 이 notebook은 silver_task 이후 실행되므로 매칭 결과가 존재해야 함

In [0]:
eval_df = spark.table(f"{CATALOG}.eval.ground_truth_case")
principals = spark.table(f"{CATALOG}.silver.silver_principals")
assets = spark.table(f"{CATALOG}.silver.silver_assets")

principal_match = (
    eval_df.alias("g")
    .join(
        principals.alias("p"),
        (F.col("g.subject_id") == F.col("p.source_principal_id"))
        & (
            ((F.col("g.subject_type") == F.lit("nhi")) & (F.col("p.principal_kind") == F.lit("NHI")))
            | ((F.col("g.subject_type") == F.lit("hi")) & (F.col("p.principal_kind") == F.lit("HI")))
        ),
        "left_anti",
    )
    .count()
)
assert principal_match == 0, f"eval rows without Silver principal match: {principal_match}"

asset_match = (
    eval_df.alias("g")
    .join(assets.alias("a"), F.col("g.asset_id") == F.col("a.source_asset_id"), "left_anti")
    .count()
)
assert asset_match == 0, f"eval rows without Silver asset match: {asset_match}"

summary = [
    "eval seed load passed",
    f"ground_truth_case: {count} rows",
    f"positive={positive_count}, distractor={distractor_count}, normal={normal_count}",
    f"eval_without_silver_principal_match: {principal_match}",
    f"eval_without_silver_asset_match: {asset_match}",
]
dbutils.notebook.exit("\n".join(summary))